In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q12/q12_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###
# Filter, join, flag and aggregate in one pipeline
total = (
    lineitem[
        (lineitem.L_RECEIPTDATE >= "1994-01-01")
        & (lineitem.L_RECEIPTDATE <  "1995-01-01")
        & (lineitem.L_COMMITDATE  <  "1995-01-01")
        & (lineitem.L_SHIPDATE   <  "1995-01-01")
        & (lineitem.L_SHIPDATE   <  lineitem.L_COMMITDATE)
        & (lineitem.L_COMMITDATE <  lineitem.L_RECEIPTDATE)
        &  lineitem.L_SHIPMODE.isin(["MAIL", "SHIP"])
    ][["L_ORDERKEY", "L_SHIPMODE"]]
    .merge(
        orders[["O_ORDERKEY", "O_ORDERPRIORITY"]],
        left_on="L_ORDERKEY", right_on="O_ORDERKEY"
    )
    .assign(
        is_urgent     = lambda df: df.O_ORDERPRIORITY.isin(["1-URGENT", "2-HIGH"]),
        is_not_urgent = lambda df: ~df.is_urgent
    )
    .groupby("L_SHIPMODE", as_index=False)
    .agg(
        g1=("is_urgent",     "sum"),
        g2=("is_not_urgent", "sum")
    )
)
print(total)